# Using SuperNeuroMAT to Code Logical Operations (NAND Gate)

When getting started using SuperNeuroMAT, it is useful to analyze basic problems and solve them using the neuron functions within SuperNeuroMAT that you will use later to complete more complex projects. Moving past the most basic logic gates of AND and OR, we can move on to the slightly more complex NAND and NOR.

For those unfamiliar with these operations, they are defined as:
<br>NAND Gate: Given a set of two inputs, return true if AT LEAST ONE of the inputs is false
<br>NOR gate: Given a set of two inputs, returns true if ONLY ONE of the inputs is true

In terms of application in the world of coding, here are the truth tables for each gate given x and y. 1 designates true, 0 designates false.

<br> For the NAND gate given two inputs:
<br>![diagram of NAND_gate](img/logic_gates_images/nand_gate_diagram.png "NAND_gate")
<br> For the NOR gate given two inputs:
<br>![diagram of NOR_gate](img/logic_gates_images/nor_gate_diagram.png "NOR_gate")

For this tutorial, we will focus on the NAND gate specifically.

In [85]:
### First, install and import superneuromat
#  use [pip install superneuromat inside of your terminal]
import superneuromat as snm
NAND_Gate=snm.SNN()
### Run this code to see the parameters of a neuron in SNM ###
help(NAND_Gate.create_neuron)

Help on method create_neuron in module superneuromat.neuromorphicmodel:

create_neuron(
    threshold: float = 0.0,
    leak: float = inf,
    reset_state: float = 0.0,
    refractory_period: int = 0,
    refractory_state: int = 0,
    initial_state: float | None = 0.0
) -> Neuron method of superneuromat.neuromorphicmodel.SNN instance
    Create a neuron in the SNN.

    Parameters
    ----------
    threshold : float, default=0.0
        Neuron threshold; the neuron spikes if its internal state is strictly
        greater than the neuron threshold
    leak : float, default=numpy.inf
        Neuron leak; the amount by which the internal state of the neuron is
        pushed towards its reset state
    reset_state : float, default=0.0
        Reset state of the neuron; the value assigned to the internal state
        of the neuron after spiking
    refractory_period : int, default=0
        Refractory period of the neuron; the number of time steps for which
        the neuron remains in

In our creation of these logic gates, we will mainly use threshold and weights to determine the function of the system. Familiarize yourself with the functions of each of these. Now that you have SNM installed, we can move on to creating the neural network. For this project, we will have two input neurons and one output neuron. The input neurons will receive either a 0 or 1 from the user, with a 1 denoting a "spike". They will then communicate their information via "synapses" to the output neuron.
<br>For the NAND Gate, we will need 5 total neurons and six synapses. We will start creating the neurons as the next step. As shown in the help function you called, neurons have a variety of important features. The input neurons need a threshold of 0, as they should spike no matter what input they receive. The NAND Gate should spike if no more than one neuron spikes. Out of those 5 neurons, one of them will be designated to cancelling the output if both neurons spike. As the cancelling step will take one time step, we will need to delay the input neurons' connections to the outputs. Another will be a constant spike so that, when neither input spikes, we can still have an output spike.

<img src = "AND_gate.png" width = "200">

Here is a diagram of the neurons, their thresholds, and their connections. The numbers indicate the indices you will view when you print the neurons.

## STEP 1: Create Neurons

In [86]:
NAND_Gate.reset
inputs=[]
outputs=[]
###First, we create the two input neurons with threshold 0 and add them to the inputs list
for i in range(2):
    id=NAND_Gate.create_neuron(threshold=0)
    inputs.append(id)


###Next, we create the final output neuron with threshold 0 and adds it to the outputs list
id=NAND_Gate.create_neuron(threshold=0)
outputs.append(id)

###Then, we have to create the cancel neuron. This one must have a threshold of 1 so it only spikes if both inputs are activated

cancel=NAND_Gate.create_neuron(threshold=1)

###For the constant-spiking neuron, we want it to spike no matter what, so we can set its threshold to -1.
# However, to add clarity, we make the refractory period 2 so that it will only spike every third time step

constant=NAND_Gate.create_neuron(threshold=-1,refractory_period=2)

###Finally, we simulate the neural network and prints out the aspects of it
print(NAND_Gate)



SNN with 5 neurons and 0 synapses @ 0x2482c73f0e0
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (5):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 []
     1           0           0         inf          0           0 []
     2           0           0         inf          0           0 []
     3           0           1         inf          0           0 []
     4           0          -1         inf          0           2 []

Synapse Info (0):
    idx    pre ->   post      weight    delay stdp_enabled


Input Spikes (0) for 0 time steps:
 Time:  Spike-value    Destination


Spike Train:
 t|id 
0 spikes since last reset


## STEP 2: Create Synapses

Next, we will take care of the synapses. For each synapse, we need to indicate the sending neuron, the receiving neuron, and the weight of the connection. All the first stage synapses (inputs and constant) have to have delay of 2 to allow the cancel neuron to work. The cancel neuron has a negative weight so that it negates all other spikes.

In [87]:
for i in range(2):
    #Creates synapses between the inputs and the output
    NAND_Gate.create_synapse(inputs[i],outputs[0],1,delay=2)
    #Creates synapses between the inputs and the cancel neuron
    NAND_Gate.create_synapse(inputs[i],cancel,1)

#Creates a synapse between the cancel neuron and the output
NAND_Gate.create_synapse(cancel,outputs[0],-5)

#Creates a synapse between the constant-spikingt neuron and the output
NAND_Gate.create_synapse(constant,outputs[0],1,delay=2)


print(NAND_Gate)


SNN with 8 neurons and 9 synapses @ 0x2482c73f0e0
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (8):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 []
     1           0           0         inf          0           0 []
     2           0           0         inf          0           0 []
     3           0           1         inf          0           0 []
     4           0          -1         inf          0           2 []
     5           0           0         inf          0           0 []
     6           0           0         inf          0           0 []
     7           0           0         inf          0           0 []

Synapse Info (9):
    idx    pre ->   post      weight    delay stdp_enabled
      0      0 ->      5           1       1 -
      1      5 ->      2           1     - 2 -
      2      0 ->      3           1       1 -
      3      1 

## STEP 3: Add Spikes

Finally, we need to add the spikes. These spikes will be 1's. As we are working on the NAND gate, if no more than one neuron spikes, the output neuron will also spike.  Once the spikes are added, we need to simulate the program to see its function. As each transfer from neuron to synapse takes one time step, we need to simulate 12 total time steps (starting at 0, gives us simulate(12)) to see every combination of inputs and outputs for the program. The first number within the add_spike function in this case defines the total time steps, the middle input is the neuron which will receive the spike, and the final number is the value of the spike.
<br>To see the results, look at the first three neurons in the spike train. The first two (indices 0,1) are the inputs and the third (index 2) is the outputs. This train should match the tables from the beginning of this tutorial

In [88]:
NAND_Gate.add_spike(0,inputs[0],0)
NAND_Gate.add_spike(0,inputs[1],0)

NAND_Gate.add_spike(3,inputs[0],1)
NAND_Gate.add_spike(3,inputs[1],0)

NAND_Gate.add_spike(6,inputs[0],0)
NAND_Gate.add_spike(6,inputs[1],1)

NAND_Gate.add_spike(9,inputs[0],1)
NAND_Gate.add_spike(9,inputs[1],1)

NAND_Gate.simulate(12) 
print(NAND_Gate)


SNN with 8 neurons and 9 synapses @ 0x2482c73f0e0
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (8):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 [---┴⋯---┴--]
     1           0           0         inf          0           0 [----⋯┴--┴--]
     2          -2           0         inf          0           0 [--┴-⋯--┴---]
     3           0           1         inf          0           0 [----⋯----┴-]
     4           0          -1         inf          0           2 [┴--┴⋯┴--┴--]
     5           0           0         inf          0           0 [----⋯----┴-]
     6           0           0         inf          0           0 [----⋯-┴--┴-]
     7           0           0         inf          0           0 [-┴--⋯-┴--┴-]

Synapse Info (9):
    idx    pre ->   post      weight    delay stdp_enabled
      0      0 ->      5           1       1 -
      1      5 ->    

If you have not already done so, try to create the NOR gate using a similar process. Put your code in the box below and run it to check the spike train to see if it's working properly. Remember, the only difference is that the NOR Gate requires both input neurons to fire for the output. Try the NOR gate tutorial, also within this GitHub afterward to see the solution.

In [89]:
###Put your code below